# SMA Module 4 — Trading System

Fewer dummies, random-forest thresholds, and a stricter growth label. Run cells from top to bottom. The parquet stays on E:. The forest fit takes a few minutes.


## 0) Imports


In [ ]:
import os
# keep caches on E: — C: fills up fast on this machine
os.environ.setdefault("MPLCONFIGDIR", r"E:\IT_SPACES\AI\.cache\matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", r"E:\IT_SPACES\AI\.cache")
os.environ.setdefault("TEMP", r"E:\IT_SPACES\AI\.cache\tmp")
os.environ.setdefault("TMP", r"E:\IT_SPACES\AI\.cache\tmp")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

DATA = Path(r"E:\IT_SPACES\AI\ZoomCamp\SMA\04\trading_strategy_and_simulation\stocks_df_combined_2026_09_18.parquet.brotli")


def clean_dataframe_from_inf_and_nan(df):
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(0, inplace=True)
    return df


def temporal_split(df, min_date, max_date, train_prop=0.7, val_prop=0.15):
    train_end = min_date + pd.Timedelta(days=(max_date - min_date).days * train_prop)
    val_end = train_end + pd.Timedelta(days=(max_date - min_date).days * val_prop)
    date = df["Date"]
    split = np.where(date <= train_end, "train", np.where(date <= val_end, "validation", "test"))
    out = df.copy()
    out["split"] = split
    return out


def load_base():
    df_full = pd.read_parquet(DATA)
    growth = [g for g in df_full.keys() if g.find("growth_") == 0 and g.find("future") < 0]
    technical_indicators = [
        "adx", "adxr", "apo", "aroon_1", "aroon_2", "aroonosc",
        "bop", "cci", "cmo", "dx", "macd", "macdsignal", "macdhist", "macd_ext",
        "macdsignal_ext", "macdhist_ext", "macd_fix", "macdsignal_fix",
        "macdhist_fix", "mfi", "minus_di", "mom", "plus_di", "dm", "ppo",
        "roc", "rocp", "rocr", "rocr100", "rsi", "slowk", "slowd", "fastk",
        "fastd", "fastk_rsi", "fastd_rsi", "trix", "ultosc", "willr",
        "ad", "adosc", "obv", "atr", "natr", "ht_dcperiod", "ht_dcphase",
        "ht_phasor_inphase", "ht_phasor_quadrature", "ht_sine_sine", "ht_sine_leadsine",
        "ht_trendmod", "avgprice", "medprice", "typprice", "wclprice",
    ]
    patterns = [g for g in df_full.keys() if g.find("cdl") >= 0]
    macro = ["gdppot_us_yoy", "gdppot_us_qoq", "cpi_core_yoy", "cpi_core_mom", "FEDFUNDS", "DGS1", "DGS5", "DGS10"]
    frame = df_full[df_full.Date >= "2000-01-01"].copy()
    frame["ln_volume"] = np.log(frame.Volume.to_numpy() + 1e-6)
    frame["div_payout"] = (frame.Dividends > 0).astype(int)
    frame["stock_split"] = (frame["Stock Splits"] > 0).astype(int)
    custom = ["SMA10", "SMA20", "growing_moving_average", "high_minus_low_relative", "volatility", "ln_volume", "div_payout", "stock_split"]
    numerical = growth + technical_indicators + patterns + custom + macro
    # Month is already a month-start timestamp, so format='%m' does not apply
    if pd.api.types.is_datetime64_any_dtype(frame["Month"]):
        frame["Month"] = frame["Month"].dt.strftime("%B")
    else:
        frame["Month"] = pd.to_datetime(frame["Month"], format="%m").dt.strftime("%B")
    frame["Weekday"] = frame["Weekday"].astype("string")
    frame["wom"] = ((frame.Date.dt.day - 1) // 7 + 1).astype(str)
    frame["month_wom"] = frame.Month + "_w" + frame.wom
    del frame["wom"]
    return frame, numerical


def with_dummies(df, numerical, drop_ticker_and_month_wom, growth_cutoff):
    frame = df.copy()
    frame["is_positive_growth_30d_future"] = (frame["growth_future_30d"] > growth_cutoff).astype(int)
    categorical = ["Month", "Weekday", "Ticker", "ticker_type", "month_wom"]
    # the homework says ticker; the column is Ticker
    if drop_ticker_and_month_wom:
        categorical = [c for c in categorical if c not in ("Ticker", "ticker", "month_wom")]
    dummies = pd.get_dummies(frame[categorical], dtype="int32")
    features = numerical + dummies.keys().to_list()
    frame = pd.concat([frame, dummies], axis=1)
    frame = temporal_split(frame, frame.Date.min(), frame.Date.max())
    return frame, features


def matrices(frame, features):
    cols = features + ["is_positive_growth_30d_future"]
    train_valid = clean_dataframe_from_inf_and_nan(frame[frame.split.isin(["train", "validation"])][cols].copy())
    all_rows = clean_dataframe_from_inf_and_nan(frame[cols].copy())
    y = train_valid.pop("is_positive_growth_30d_future")
    all_rows.pop("is_positive_growth_30d_future")
    return train_valid, y, all_rows


def simulate(frame, pred):
    # fees are 0.2% for the round trip; capital is $100 x 30 days x the 75% daily count
    test_pos = (frame.split == "test") & (frame[pred] == 1)
    count = int(test_pos.sum())
    gross = float((frame.loc[test_pos, pred] * 100 * (frame.loc[test_pos, "growth_future_30d"] - 1)).sum())
    fees = float((-frame.loc[test_pos, pred] * 100 * 0.002).sum())
    net = gross + fees
    daily = frame.loc[test_pos].groupby("Date")[pred].count()
    avg = float(daily.mean())
    q75 = float(daily.quantile(0.75))
    capital = 100 * 30 * q75
    cagr = ((capital + net) / capital) ** (1 / 4)
    return count, avg, cagr


## Q1 — Less features

Drop `Ticker` and `month_wom` just before `get_dummies`. The lesson count is 301 features. This setup should be 208. Then read the CAGR of the 0.55 random-forest rule.


In [ ]:
base, numerical = load_base()
q1_frame, q1_features = with_dummies(base, numerical, True, 1)
print("features", len(q1_features))
x_tv, y_tv, x_all = matrices(q1_frame, q1_features)
rf = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=-1)
rf.fit(x_tv, y_tv)
proba = rf.predict_proba(x_all)[:, 1]
q1_frame["pred10_rf_best_rule_55"] = (proba >= 0.55).astype(int)
trades, avg, cagr = simulate(q1_frame, "pred10_rf_best_rule_55")
# 1.128 is not a form choice. The nearest is 1.126.
print("Q1", trades, avg, cagr)


features 208
Q1 3756 6.058064516129032 1.1278789977008685


## Q2 — Fewer than 5 trades a day

Same forest as Q1. Keep the threshold whose average daily trades are under 5 and whose CAGR is the best in that group.


In [ ]:
rows = []
for threshold in [0.55, 0.56, 0.57, 0.58, 0.59, 0.60, 0.61, 0.62, 0.63, 0.64, 0.65]:
    name = f"pred_rf_{threshold:.2f}"
    q1_frame[name] = (proba >= threshold).astype(int)
    trades, avg, cagr = simulate(q1_frame, name)
    rows.append((threshold, trades, avg, cagr))
    print(f"{threshold:.2f} trades={trades} avg_per_day={avg:.3f} cagr={cagr:.6f}")
# 0.56 still averages more than 5 trades a day. 0.57 is the first one under 5. Rounding 4.85 prints as 5.
under = [row for row in rows if row[2] < 5]
best = max(under, key=lambda row: row[3])
print("Q2", best[0])


0.55 trades=3756 avg_per_day=6.058 cagr=1.127879
0.56 trades=2780 avg_per_day=5.346 cagr=1.101348
0.57 trades=2086 avg_per_day=4.851 cagr=1.091977
0.58 trades=1519 avg_per_day=4.494 cagr=1.067135
0.59 trades=1090 avg_per_day=4.325 cagr=1.057418
0.60 trades=745 avg_per_day=3.942 cagr=1.049010
0.61 trades=522 avg_per_day=3.600 cagr=1.054267
0.62 trades=349 avg_per_day=3.490 cagr=1.033463
0.63 trades=229 avg_per_day=3.095 cagr=1.022911
0.64 trades=153 avg_per_day=3.060 cagr=1.017595
0.65 trades=92 avg_per_day=3.286 cagr=1.009360
Q2 0.57


## Q3 — Growth above 1.022

Start again from the full dummy list. A positive label now means 30-day growth above 1.022.
The form asks for the trade count of `pred5_clf_10`, but the choices are 6 / 7 / 8 / 9,
so I report average trades per day.


In [ ]:
# fresh dummy set: Q3 does not keep the reduced features from Q1
q3_frame, q3_features = with_dummies(base, numerical, False, 1.022)
print("features", len(q3_features), "pos_rate", q3_frame["is_positive_growth_30d_future"].mean())
x_tv, y_tv, x_all = matrices(q3_frame, q3_features)
clf = DecisionTreeClassifier(max_depth=10, random_state=42)
clf.fit(x_tv, y_tv)
q3_frame["pred5_clf_10"] = clf.predict(x_all)
trades, avg, cagr = simulate(q3_frame, "pred5_clf_10")
print("Q3 avg_per_day", round(avg), "raw_avg", avg, "total_trades", trades, "cagr", cagr)
# form choices are 6/7/8/9 — that is avg trades/day, not the 7337 total


features 301 pos_rate 0.49120578174124535
Q3 avg_per_day 8 raw_avg 8.009825327510917 total_trades 7337 cagr 1.1217904249739772


## Q4 — Ideal strategy

I would not put a flat $100 on every positive flag. Each morning I would take the reduced-dummy
random forest, keep names with probability at least 0.57, rank them by probability, and invest
only in the top three that day. That day's capital would be split equally across those names,
so a quiet day does not leave cash idle and a crowded day does not force a long list of tiny bets.
If fewer than two names clear 0.57, I would skip the day. I would treat agreement with the
depth-10 tree from the 1.022 growth label as a soft filter: when both models agree I would allow
a slightly larger size, and when they disagree I would skip. Round-trip fees stay at 0.2% in the
simulation so the CAGR stays honest.
